In [3]:
"""
Warehouse Location Heuristic — Scenario 1  (v2 — corrected plant coverage)
===========================================================================
BUG FIXED: v1 incorrectly assumed all products are available at all plants
           when computing baseline plant coverage. In reality:
               Plant 1 → Product 1 only
               Plant 2 → Product 2 only
               Plant 3 → Product 3 only
               Plant 4 → Products 4 and 5 only

FIX APPROACH:
  Step A. For each (customer, product) pair, check if the product's SOURCE
          plant is within 500 miles. If yes → already covered by plant,
          store as a "plant-covered pair".
  Step B. Compute plant coverage % using only these valid pairs.
  Step C. For warehouse optimisation, subtract plant-covered demand from
          the total. The heuristic only needs to cover the remaining gap.
  Step D. Greedy loop runs on UNCOVERED demand only (correct gap).

Outputs (Excel workbook — 5 sheets):
  1. Executive Summary
  2. Warehouses           — lat/long, city, demand covered
  3. Customer Assignment  — nearest facility per customer
  4. Coverage Trace       — cumulative coverage after each warehouse
  5. Cost Comparison      — before vs after transport cost
"""

import pandas as pd
import numpy as np

# ── 0. CONFIG ──────────────────────────────────────────────────────────────────
EXCEL_PATH      = '/content/drive/MyDrive/Decision Spot/OR Case Study - Decision Spot.xlsx'
OUTPUT_PATH     = '/content/drive/MyDrive/Decision Spot/warehouse_heuristic_v2_output.xlsx'

COVERAGE_RADIUS  = 500    # miles — service standard
COVERAGE_TARGET  = 0.80   # 80% of total demand must be within radius
COST_PER_TRUCK   = 2      # $ per truck per mile
TRUCK_CAPACITY   = 10     # tons per truck
BASE_YEAR        = 2012   # planning year

# Current production assignments (product → source plant)
PRODUCT_PLANT = {1: 1, 2: 2, 3: 3, 4: 4, 5: 4}

# ── 1. LOAD DATA ───────────────────────────────────────────────────────────────
print("Loading data...")
xl = pd.ExcelFile(EXCEL_PATH)

customers_df = xl.parse('Customers')
demand_df    = xl.parse('Demand')
dist_raw     = xl.parse('Distances')

# ── 2. PARSE DISTANCE MATRICES ─────────────────────────────────────────────────
# Left block: Plant → Customer
plant_cust_raw = dist_raw[['Plant ID', 'Customer ID', 'Distance']].dropna()
plant_cust_raw.columns = ['plant_id', 'customer_id', 'distance_mi']
plant_cust_raw = plant_cust_raw.astype({'plant_id': int, 'customer_id': int})
plant_cust_matrix = plant_cust_raw.pivot(
    index='plant_id', columns='customer_id', values='distance_mi')

# Right block: Customer → Customer (full 50×50 matrix)
cust_cust_raw = dist_raw[['Customer ID.1', 'Customer ID.2', 'Distance.1']].dropna()
cust_cust_raw.columns = ['from_customer', 'to_customer', 'distance_mi']
cust_cust_raw = cust_cust_raw.astype({'from_customer': int, 'to_customer': int})
cust_cust_matrix = cust_cust_raw.pivot(
    index='from_customer', columns='to_customer', values='distance_mi')

# ── 3. FILTER TO BASE YEAR DEMAND ─────────────────────────────────────────────
demand_2012 = demand_df[demand_df['Time Period'] == BASE_YEAR].copy()
demand_2012['source_plant'] = demand_2012['Product ID'].map(PRODUCT_PLANT)

total_demand_tons = demand_2012['Demand (in tonnes)'].sum()
target_tons       = COVERAGE_TARGET * total_demand_tons

print(f"Total annual demand ({BASE_YEAR}) : {total_demand_tons:,.1f} tons")
print(f"80% coverage target               : {target_tons:,.1f} tons")

# ── 4. STEP A — IDENTIFY PLANT-COVERED (customer, product) PAIRS ───────────────
# A pair is covered by its source plant only if that plant is within 500 miles.
# This correctly enforces that Plant 1 can only cover Product 1 demand, etc.

plant_covered_pairs = []
plant_covered_set   = set()   # (customer_id, product_id) tuples

for _, row in demand_2012.iterrows():
    cid   = int(row['Customer ID'])
    pid   = int(row['Product ID'])
    plant = int(row['source_plant'])
    tons  = row['Demand (in tonnes)']
    dist  = plant_cust_matrix.loc[plant, cid]

    if dist <= COVERAGE_RADIUS:
        plant_covered_pairs.append({
            'customer_id'  : cid,
            'product_id'   : pid,
            'source_plant' : plant,
            'distance_mi'  : round(dist, 2),
            'covered_tons' : tons
        })
        plant_covered_set.add((cid, pid))

plant_covered_df   = pd.DataFrame(plant_covered_pairs)
plant_covered_tons = plant_covered_df['covered_tons'].sum()
plant_coverage_pct = plant_covered_tons / total_demand_tons * 100

print(f"\n--- Baseline plant coverage (corrected) ---")
print(f"Plant-covered tons  : {plant_covered_tons:,.1f}  ({plant_coverage_pct:.1f}%)")
print(f"Unique customers covered by at least one plant-product: "
      f"{plant_covered_df['customer_id'].nunique()}")

if plant_coverage_pct >= COVERAGE_TARGET * 100:
    print("Plants alone achieve 80% — no warehouses needed!")

# ── 5. STEP B — COMPUTE RESIDUAL (UNCOVERED) DEMAND PER CUSTOMER ──────────────
# Subtract all plant-covered (customer, product) pairs from total demand.
# The remaining demand per customer is what warehouses must cover.

demand_2012['plant_covered'] = demand_2012.apply(
    lambda r: (int(r['Customer ID']), int(r['Product ID'])) in plant_covered_set,
    axis=1
)

# Residual demand = demand NOT covered by any plant
residual_demand = (
    demand_2012[~demand_2012['plant_covered']]
    .groupby('Customer ID')['Demand (in tonnes)']
    .sum()
    .reset_index()
    .rename(columns={'Customer ID': 'customer_id', 'Demand (in tonnes)': 'residual_tons'})
)

# Total demand per customer (for assignment table)
total_per_customer = (
    demand_2012.groupby('Customer ID')['Demand (in tonnes)']
    .sum()
    .reset_index()
    .rename(columns={'Customer ID': 'customer_id', 'Demand (in tonnes)': 'annual_tons'})
)

# ── 6. MERGE INTO CUSTOMER TABLE ───────────────────────────────────────────────
customers = customers_df[['ID', 'City', 'State', 'Latitude', 'Longitude']].copy()
customers.columns = ['customer_id', 'city', 'state', 'lat', 'lon']
customers = (customers
             .merge(total_per_customer, on='customer_id')
             .merge(residual_demand, on='customer_id', how='left'))
customers['residual_tons'] = customers['residual_tons'].fillna(0)

# Flag: customer has any residual demand left for warehouses to cover
customers['needs_warehouse_coverage'] = customers['residual_tons'] > 0

print(f"\nCustomers with residual demand (need warehouse): "
      f"{customers['needs_warehouse_coverage'].sum()}")
print(f"Residual demand to cover: {customers['residual_tons'].sum():,.1f} tons")

# ── 7. STEP C — GREEDY WAREHOUSE PLACEMENT ON RESIDUAL DEMAND ─────────────────
# The heuristic now operates on residual_tons (uncovered by plants).
# A warehouse covers a customer's residual demand if it is within 500 miles.
# We stop when:
#   plant_covered_tons + warehouse_covered_tons >= 80% of total_demand_tons

print("\nRunning greedy heuristic on residual demand...")

# Boolean mask: True = this customer's residual demand is still uncovered
# Index-aligned to customers dataframe
uncovered_mask  = customers['needs_warehouse_coverage'].values.copy()
candidate_ids   = customers['customer_id'].tolist()
cust_idx        = {cid: i for i, cid in enumerate(customers['customer_id'])}
residual_arr    = customers['residual_tons'].values

warehouses_placed = []
coverage_trace    = []

iteration = 0
while True:
    # Covered so far = plant coverage + warehouse coverage accumulated
    wh_covered_tons      = residual_arr[~uncovered_mask & customers['needs_warehouse_coverage'].values].sum()
    total_covered_tons   = plant_covered_tons + wh_covered_tons
    current_pct          = total_covered_tons / total_demand_tons * 100

    coverage_trace.append({
        'warehouses_placed'  : iteration,
        'plant_covered_tons' : round(plant_covered_tons, 1),
        'wh_covered_tons'    : round(wh_covered_tons, 1),
        'total_covered_tons' : round(total_covered_tons, 1),
        'coverage_pct'       : round(current_pct, 2),
        'remaining_gap_tons' : round(total_demand_tons - total_covered_tons, 1)
    })

    if total_covered_tons >= target_tons:
        print(f"\nTarget reached after {iteration} warehouse(s)!")
        print(f"Final coverage: {current_pct:.2f}%")
        break

    # Score every candidate: residual tons it would newly cover
    best_score    = -1
    best_cand_id  = None
    best_new_mask = None

    for cand_id in candidate_ids:
        dists = cust_cust_matrix.loc[cand_id]
        newly_covered = np.array([
            uncovered_mask[cust_idx[cid]] and
            customers.iloc[cust_idx[cid]]['needs_warehouse_coverage'] and
            (dists.get(cid, np.inf) <= COVERAGE_RADIUS)
            for cid in customers['customer_id']
        ])
        score = residual_arr[newly_covered].sum()
        if score > best_score:
            best_score    = score
            best_cand_id  = cand_id
            best_new_mask = newly_covered

    if best_cand_id is None or best_score == 0:
        print("Warning: no candidate can improve coverage further.")
        break

    iteration += 1
    wh_row = customers[customers['customer_id'] == best_cand_id].iloc[0]
    cum_pct = (plant_covered_tons + wh_covered_tons + best_score) / total_demand_tons * 100

    warehouses_placed.append({
        'warehouse_id'            : iteration,
        'customer_site_id'        : best_cand_id,
        'city'                    : wh_row['city'],
        'state'                   : wh_row['state'],
        'latitude'                : wh_row['lat'],
        'longitude'               : wh_row['lon'],
        'newly_covered_tons'      : round(best_score, 1),
        'cumulative_coverage_pct' : round(cum_pct, 2)
    })

    uncovered_mask = uncovered_mask & ~best_new_mask

    print(f"  WH{iteration}: {wh_row['city']}, {wh_row['state']}"
          f"  +{best_score:,.0f} residual tons"
          f"  ({cum_pct:.1f}% total cumulative)")

# ── 8. OUTPUT DF 1: Warehouses ─────────────────────────────────────────────────
warehouses_df = pd.DataFrame(warehouses_placed)

# ── 9. OUTPUT DF 2: Customer Assignment ───────────────────────────────────────
print("\nBuilding customer assignment table...")

def assign_customer(row):
    cid       = row['customer_id']
    # Start with nearest plant (for this customer's products)
    # Use overall min plant distance as proxy for nearest source
    best_dist = plant_cust_matrix[cid].min()
    best_type = 'Plant'
    best_id   = int(plant_cust_matrix[cid].idxmin())

    for wh in warehouses_placed:
        d = cust_cust_matrix.loc[wh['customer_site_id'], cid]
        if d < best_dist:
            best_dist = d
            best_type = 'Warehouse'
            best_id   = wh['warehouse_id']

    return pd.Series({
        'nearest_facility_type': best_type,
        'nearest_facility_id'  : best_id,
        'nearest_distance_mi'  : round(best_dist, 1),
        'within_500mi'         : best_dist <= COVERAGE_RADIUS
    })

assign_extra = customers.apply(assign_customer, axis=1)
customer_assign_df = pd.concat([
    customers[['customer_id', 'city', 'state', 'lat', 'lon',
               'annual_tons', 'residual_tons', 'needs_warehouse_coverage']],
    assign_extra
], axis=1)

# ── 10. OUTPUT DF 3: Coverage Trace ───────────────────────────────────────────
coverage_summary_df = pd.DataFrame(coverage_trace)

# ── 11. OUTPUT DF 4: Cost Comparison ──────────────────────────────────────────
print("Computing cost comparison...")

# BEFORE: every (customer, product) served direct from source plant
before_rows = []
for _, row in demand_2012.iterrows():
    cid   = int(row['Customer ID'])
    plant = int(row['source_plant'])
    tons  = row['Demand (in tonnes)']
    dist  = plant_cust_matrix.loc[plant, cid]
    cost  = np.ceil(tons / TRUCK_CAPACITY) * dist * COST_PER_TRUCK
    before_rows.append({'customer_id': cid, 'before_cost': cost})

before_by_cust = (
    pd.DataFrame(before_rows)
    .groupby('customer_id')['before_cost'].sum()
    .reset_index()
)

# AFTER: plant-covered pairs unchanged; warehouse-covered get two-leg cost
after_rows = []
for _, crow in customer_assign_df.iterrows():
    cid      = crow['customer_id']
    fac_type = crow['nearest_facility_type']
    fac_id   = crow['nearest_facility_id']
    near_d   = crow['nearest_distance_mi']
    ann_tons = crow['annual_tons']

    before_c = before_by_cust[before_by_cust['customer_id'] == cid]['before_cost'].values
    before_c = before_c[0] if len(before_c) else 0

    if fac_type == 'Plant':
        after_cost = before_c   # no change — same direct route
    else:
        wh_site = warehouses_placed[fac_id - 1]['customer_site_id']
        # Inbound: each product from its source plant to the warehouse site
        inbound = 0
        for prod_id, plant_id in PRODUCT_PLANT.items():
            prod_tons = demand_2012[
                (demand_2012['Customer ID'] == cid) &
                (demand_2012['Product ID'] == prod_id)
            ]['Demand (in tonnes)'].sum()
            if prod_tons > 0:
                d_pw = plant_cust_matrix.loc[plant_id, wh_site]
                inbound += np.ceil(prod_tons / TRUCK_CAPACITY) * d_pw * COST_PER_TRUCK
        # Outbound: warehouse to customer (combined tonnage)
        outbound   = np.ceil(ann_tons / TRUCK_CAPACITY) * near_d * COST_PER_TRUCK
        after_cost = inbound + outbound

    saving = before_c - after_cost
    after_rows.append({
        'customer_id'           : cid,
        'city'                  : crow['city'],
        'state'                 : crow['state'],
        'annual_tons'           : round(ann_tons, 1),
        'served_by'             : fac_type,
        'nearest_distance_mi'   : near_d,
        'before_transport_cost' : round(before_c, 0),
        'after_transport_cost'  : round(after_cost, 0),
        'cost_saving'           : round(saving, 0),
        'saving_pct'            : round(saving / before_c * 100, 1) if before_c > 0 else 0
    })

cost_comparison_df = pd.DataFrame(after_rows)

# ── 12. PRINT SUMMARY ──────────────────────────────────────────────────────────
total_before = cost_comparison_df['before_transport_cost'].sum()
total_after  = cost_comparison_df['after_transport_cost'].sum()
net_saving   = total_before - total_after

print("\n" + "="*62)
print("CORRECTED HEURISTIC RESULTS SUMMARY")
print("="*62)
print(f"Plant-only coverage (corrected) : {plant_coverage_pct:.1f}%")
print(f"Warehouses needed               : {len(warehouses_df)}")
print(f"Final total coverage            : {coverage_summary_df['coverage_pct'].iloc[-1]:.2f}%")
print(f"\nWarehouse locations:")
for _, w in warehouses_df.iterrows():
    print(f"  WH{int(w['warehouse_id'])}: {w['city']}, {w['state']}"
          f"  lat={w['latitude']:.4f}  lon={w['longitude']:.4f}"
          f"  cumulative={w['cumulative_coverage_pct']}%")
print(f"\nTransport cost BEFORE : ${total_before:>15,.0f}")
print(f"Transport cost AFTER  : ${total_after:>15,.0f}")
print(f"Net change            : ${net_saving:>15,.0f}  ({net_saving/total_before*100:.1f}%)")

# ── 13. WRITE OUTPUT EXCEL ─────────────────────────────────────────────────────
print("\nWriting output Excel...")
with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:

    pd.DataFrame({
        'Metric': [
            'Total annual demand (tons)', '80% target (tons)',
            'Plant coverage — v1 (WRONG) %', 'Plant coverage — v2 (CORRECT) %',
            'Bug explanation',
            'Warehouses needed', 'Final coverage (%)',
            'Transport cost BEFORE ($)', 'Transport cost AFTER ($)',
            'Net cost change ($)', 'Net cost change (%)'
        ],
        'Value': [
            round(total_demand_tons, 1), round(target_tons, 1),
            '41.3% — assumed all products at all plants',
            round(plant_coverage_pct, 2),
            'v1 used min distance across all 4 plants; v2 checks only the source plant for each product',
            len(warehouses_df),
            round(coverage_summary_df['coverage_pct'].iloc[-1], 2),
            round(total_before, 0), round(total_after, 0),
            round(net_saving, 0),
            round(net_saving / total_before * 100, 2)
        ]
    }).to_excel(writer, sheet_name='Executive Summary', index=False)

    warehouses_df.to_excel(writer, sheet_name='Warehouses', index=False)
    customer_assign_df.to_excel(writer, sheet_name='Customer Assignment', index=False)
    coverage_summary_df.to_excel(writer, sheet_name='Coverage Trace', index=False)
    cost_comparison_df.to_excel(writer, sheet_name='Cost Comparison', index=False)
    plant_covered_df.to_excel(writer, sheet_name='Plant Covered Pairs', index=False)

print(f"Saved → {OUTPUT_PATH}")

Loading data...
Total annual demand (2012) : 415,728.7 tons
80% coverage target               : 332,582.9 tons

--- Baseline plant coverage (corrected) ---
Plant-covered tons  : 45,749.9  (11.0%)
Unique customers covered by at least one plant-product: 21

Customers with residual demand (need warehouse): 50
Residual demand to cover: 369,978.7 tons

Running greedy heuristic on residual demand...
  WH1: Roxbury, NY  +99,276 residual tons  (34.9% total cumulative)
  WH2: Reno, NV  +98,817 residual tons  (58.7% total cumulative)
  WH3: AMARILLO, TX  +81,195 residual tons  (78.2% total cumulative)
  WH4: Boise, ID  +53,963 residual tons  (91.2% total cumulative)

Target reached after 4 warehouse(s)!
Final coverage: 91.17%

Building customer assignment table...
Computing cost comparison...

CORRECTED HEURISTIC RESULTS SUMMARY
Plant-only coverage (corrected) : 11.0%
Warehouses needed               : 4
Final total coverage            : 91.17%

Warehouse locations:
  WH1: Roxbury, NY  lat=42.287